In [1]:
import requests
import pandas as pd
# Azért, hogy pd.Dataframe-nél ne csak az általa választott oszlopmennyiséget adja meg, ezért szükség van az alábbira:
pd.set_option("display.max_columns",30)
from datetime import datetime

# napok közit különbség kiszámolása
from dateutil.relativedelta import relativedelta

In [ ]:
"""
    Steps for the Transform Load Lambda function
    1. Get data from S3 (taxi, weather)
    2. Weather data transformations
    3. Taxi data transformations --> Done
    4. Update dim_payment_type
    5. Update dim_company
    6. Update fact_taxi_trips with the ids from the dim_payment_type and dim_company
    7. Upload dim_weather to S3
    8. Upload fact_taxi_trips to S3
    9. Upload dim_payment_type and dim_company (current and previous versions)

"""

In [2]:
current_datetime = datetime.now() - relativedelta(months=2)

formatted_datetime = current_datetime.strftime("%Y-%m-%d")

#taxi trips lekérés fix napra:

url = (
    f"https://data.cityofchicago.org/resource/ajtu-isnz.json?"
    f"$where=trip_start_timestamp >= '{formatted_datetime}T00:00:00'"
    f"AND trip_start_timestamp <= '{formatted_datetime}T23:59:59'"
    f"&$limit=30000"
)

response = requests.get(url)
data = response.json()

# Adatok df-esítése

taxi_trips = pd.DataFrame(data)

taxi_trips.head()

,trip_id,taxi_id,trip_start_timestamp,trip_end_timestamp,trip_seconds,trip_miles,pickup_community_area,fare,tips,tolls,extras,trip_total,payment_type,company,pickup_centroid_latitude,pickup_centroid_longitude,pickup_centroid_location,dropoff_community_area,dropoff_centroid_latitude,dropoff_centroid_longitude,dropoff_centroid_location,pickup_census_tract,dropoff_census_tract
0,a7b269645dda4187ad38ec758f7d3b83bfe7454d,807f82595e67582251c7abcdde792543facfa879272732...,2025-09-24T23:45:00.000,2025-09-25T00:00:00.000,991,7.07,56,39.88,8.08,0,0,48.46,Mobile,Flash Cab,41.79259236,-87.769615453,"{'type': 'Point', 'coordinates': [-87.76961545...",NaN,NaN,NaN,NaN,NaN,NaN
1,a6aa9472a7b8cb41f53ea9c05e020f5912a1dd24,bf66d006c4634bdc6c98ee6d505187ed750e647a6b4d1c...,2025-09-24T23:45:00.000,2025-09-25T00:15:00.000,1418,14.47,53,36.25,0,0,0,36.25,Prcard,Flash Cab,41.673819904,-87.635739777,"{'type': 'Point', 'coordinates': [-87.63573977...",38,41.812948939,-87.617859676,"{'type': 'Point', 'coordinates': [-87.61785967...",NaN,NaN
2,a69e4c16f5104010c5ff18d48068912417ff357d,58c784c628cc2b0b77468e6a406acea729301abbc17a62...,2025-09-24T23:45:00.000,2025-09-24T23:45:00.000,291,1.11,6,6,0,0,1,7,Cash,Taxicab Insurance Agency Llc,41.944226601,-87.655998182,"{'type': 'Point', 'coordinates': [-87.65599818...",6,41.944226601,-87.655998182,"{'type': 'Point', 'coordinates': [-87.65599818...",NaN,NaN
3,9fbfafbe6ad36fca5e0f53d4932f91da162d1d7f,c60633c5d2053dd1c2076286b88196cf83f4fe0ecc1945...,2025-09-24T23:45:00.000,2025-09-25T00:00:00.000,652,7.91,38,21.25,0,0,0,21.25,Prcard,Chicago Independents,41.812948939,-87.617859676,"{'type': 'Point', 'coordinates': [-87.61785967...",40,41.792357223,-87.61793138,"{'type': 'Point', 'coordinates': [-87.61793138...",NaN,NaN
4,9d96540d0c469860f0eca786401c749d69e8d31e,4cf389626b3d1df35aef815a964571fd5ea7dddda01e56...,2025-09-24T23:45:00.000,2025-09-25T00:15:00.000,1299,9.62,6,21.82,4.56,0,0,26.88,Mobile,Flash Cab,41.944226601,-87.655998182,"{'type': 'Point', 'coordinates': [-87.65599818...",28,41.874005383,-87.66351755,"{'type': 'Point', 'coordinates': [-87.66351754...",NaN,NaN


### Taxi data transformations 

In [ ]:
taxi_trips.drop(['pickup_census_tract' , 'dropoff_census_tract','pickup_centroid_location','dropoff_centroid_location'], axis=1, inplace=True)

#Az üres értékkel rendelkező sorok elhagyása

taxi_trips.dropna(inplace=True)

taxi_trips.rename(columns={'pickup_community_area':'pickup_community_area_id',
                            'dropoff_community_area' : 'dropoff_community_area_id'}, inplace=True )

# Taxi trips startdate adattípus objectről stringre

taxi_trips['trip_start_timestamp'] = pd.to_datetime(taxi_trips['trip_start_timestamp'])

# dátumidő segédoszlop időjáráshoz --> idő lekerekítése a legközelebbi egészórára

taxi_trips['datetime_for_weather'] = taxi_trips["trip_start_timestamp"].dt.floor('h')

In [ ]:
def taxi_trips_transformations(taxi_trips:pd.DataFrame) -> pd.DataFrame:

    """
    Perform transformations on the taxi data.
    1. Drop selected columns
    2. Drop NULL values across all columns
    3. Rename selected columns
    4. Create "datetime_for_weather" helper column (for dim_weather join).

    :param taxi_trips: The DataFrame holding the daily taxi trips.
    :raises TypeError: When taxi_trips parameter is not a valid pandas DataFrame.
    :return:           Transformed taxi trips DataFrame

    """

    #Alaphibakezelés (ez mindig kell!)
    if not isinstance(taxi_trips,pd.DataFrame):
        raise TypeError("taxi_trips is not a valid pandas DataFrame.")

    taxi_trips.drop(['pickup_census_tract' , 'dropoff_census_tract','pickup_centroid_location','dropoff_centroid_location'], axis=1, inplace=True)

    #Az üres értékkel rendelkező sorok elhagyása

    taxi_trips.dropna(inplace=True)

    taxi_trips.rename(columns={'pickup_community_area':'pickup_community_area_id',
                            'dropoff_community_area' : 'dropoff_community_area_id'}, inplace=True )

    # Taxi trips startdate adattípus objectről stringre

    taxi_trips['trip_start_timestamp'] = pd.to_datetime(taxi_trips['trip_start_timestamp'])

    # dátumidő segédoszlop időjáráshoz --> idő lekerekítése a legközelebbi egészórára

    taxi_trips['datetime_for_weather'] = taxi_trips["trip_start_timestamp"].dt.floor('h')

    return taxi_trips

In [ ]:
#teszt

taxi_trips_transformed = taxi_trips_transformations(taxi_trips)

taxi_trips_transformed.head()

,trip_id,taxi_id,trip_start_timestamp,trip_end_timestamp,trip_seconds,trip_miles,pickup_community_area_id,fare,tips,tolls,extras,trip_total,payment_type,company,pickup_centroid_latitude,pickup_centroid_longitude,dropoff_community_area_id,dropoff_centroid_latitude,dropoff_centroid_longitude,datetime_for_weather
1,a6aa9472a7b8cb41f53ea9c05e020f5912a1dd24,bf66d006c4634bdc6c98ee6d505187ed750e647a6b4d1c...,2025-09-24 23:45:00,2025-09-25T00:15:00.000,1418,14.47,53,36.25,0,0,0,36.25,Prcard,Flash Cab,41.673819904,-87.635739777,38,41.812948939,-87.617859676,2025-09-24 23:00:00
2,a69e4c16f5104010c5ff18d48068912417ff357d,58c784c628cc2b0b77468e6a406acea729301abbc17a62...,2025-09-24 23:45:00,2025-09-24T23:45:00.000,291,1.11,6,6,0,0,1,7,Cash,Taxicab Insurance Agency Llc,41.944226601,-87.655998182,6,41.944226601,-87.655998182,2025-09-24 23:00:00
3,9fbfafbe6ad36fca5e0f53d4932f91da162d1d7f,c60633c5d2053dd1c2076286b88196cf83f4fe0ecc1945...,2025-09-24 23:45:00,2025-09-25T00:00:00.000,652,7.91,38,21.25,0,0,0,21.25,Prcard,Chicago Independents,41.812948939,-87.617859676,40,41.792357223,-87.61793138,2025-09-24 23:00:00
4,9d96540d0c469860f0eca786401c749d69e8d31e,4cf389626b3d1df35aef815a964571fd5ea7dddda01e56...,2025-09-24 23:45:00,2025-09-25T00:15:00.000,1299,9.62,6,21.82,4.56,0,0,26.88,Mobile,Flash Cab,41.944226601,-87.655998182,28,41.874005383,-87.66351755,2025-09-24 23:00:00
5,9624cf5b1c4d8bbe67f7b064edc8ff8ae6188ff0,bb277fc77c865565c0fc305dcad0c6b6633e72ce0fcfac...,2025-09-24 23:45:00,2025-09-24T23:45:00.000,364,1.02,8,6,2,0,0,8.5,Credit Card,Taxicab Insurance Agency Llc,41.899602111,-87.633308037,8,41.899602111,-87.633308037,2025-09-24 23:00:00


In [ ]:
#teszt

taxi_trips_transformed.info()

<class 'pandas.core.frame.DataFrame'>
Index: 20866 entries, 1 to 23064
Data columns (total 20 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   trip_id                     20866 non-null  object        
 1   taxi_id                     20866 non-null  object        
 2   trip_start_timestamp        20866 non-null  datetime64[ns]
 3   trip_end_timestamp          20866 non-null  object        
 4   trip_seconds                20866 non-null  object        
 5   trip_miles                  20866 non-null  object        
 6   pickup_community_area_id    20866 non-null  object        
 7   fare                        20866 non-null  object        
 8   tips                        20866 non-null  object        
 9   tolls                       20866 non-null  object        
 10  extras                      20866 non-null  object        
 11  trip_total                  20866 non-null  object        


### Dim Company and Dim Payment Type update

In [ ]:
def update_dim_company_dim_payment_type(taxi_trips: pd.DataFrame, dim_df: pd.DataFrame, id_col: str, value_col=str) -> pd.DataFrame:

    """
    Extemd the dimension DataFrame with new values if there are any.

    :param taxi_trips: DataFrame with the daily taxi trips.
    :param dim_df: DataFrame with the dimension data (company, payment_type).
    :param id_col: The id column of the dimension DataFrame.
    :param value_col: Name of the column in dimension DataFrame containing the values.
    :return: The updated dimension data, if new values are in the taxi data, them will be loaded to it.

    """

    #duplikációk eltávolítása

    todays_dim_data = pd.DataFrame(taxi_trips[value_col].unique(), columns=[value_col])

    #majd kiadatjuk az összes olyan értéket a dummy táblából ami a korábban definiált dim táblában nincs benne

    new_dim_data = todays_dim_data[~todays_dim_data[value_col].isin(dim_df[value_col])]

    #majd azért, hogy ne a dimenzió táblában lévő számnál lévő értéket írja felül, hanem adjon hozzá egy új ID-t, ezért szükséges az alábbi

    if not new_dim_data.empty:
        max_id = dim_df[id_col].max()
        new_dim_data[id_col] = range(max_id+1,max_id+1+len(new_dim_data))
        dim_df = pd.concat([dim_df, new_dim_data], ignore_index = True )

    return dim_df